# Causal-Feature Audit — `ctx_lower_stnum_next3`

`AUDIT_STATUS.md` flags that the context feature **`ctx_lower_stnum_next3`** is
**not causal**: it is built from the *next three* stream rows, which are not
available when scoring the current packet. This notebook (1) demonstrates the
non-causality directly, and (2) quantifies the effect of removing it by
retraining LightGBM, XGBoost and Random Forest on:

- the **53-feature** set (with the non-causal feature — as in Table 7.11),
- the **52-feature causal** set (feature removed),
- **52 + a causal replacement** (`current stNum < recent-window max`, trailing only).

All three use the same corrected split and the same target-recall policy, so the
error counts are comparable. Trees only -> fast, deterministic. **Run All.**

## Configuration, imports, and paths

In [ ]:
import os, numpy as np, pandas as pd, joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import lightgbm as lgb, xgboost as xgb

# resolve repo paths whether run from the repo root or notebook/
_d = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(_d, "notebook/artifacts/corrected_44")): break
    _d = os.path.dirname(_d)
REPO = _d
ROOT = os.path.join(REPO, "notebook/artifacts/corrected_44")
ART  = os.path.join(REPO, "notebook/artefacts_corrected")
SEED, TEST_SIZE, VAL_SIZE, VAL_FLOOR = 42, 0.30, 0.20, 0.99990
print("repo:", REPO)


## Build the 53-feature corpus (base + 3 context + 6 guards)

In [ ]:
z = np.load(next(os.path.join(ROOT, f) for f in os.listdir(ROOT) if f.endswith(".npz")), allow_pickle=True)
X, y = z["features"], z["labels"].astype(int); names = list(z["feature_names"])
st = X[:, names.index("stNum")].astype(float); sq = X[:, names.index("sqNum")].astype(float)
S = pd.Series(st); d = np.diff(st, prepend=st[0])

ctx = {"ctx_st_back_count_w20": pd.Series((d < 0).astype(float)).rolling(20, min_periods=1).sum().values,
       "ctx_st_threads_w20":    S.rolling(20, min_periods=1).apply(lambda a: len(np.unique(a)), raw=True).values}
# NON-CAUSAL feature: uses stNum of the next 1..3 rows
fwd = np.full(len(st), np.inf)
for k in (1, 2, 3):
    sh = np.full(len(st), np.inf); sh[:-k] = st[k:]; fwd = np.minimum(fwd, sh)
ctx["ctx_lower_stnum_next3"] = ((fwd < st) & np.isfinite(fwd)).astype(float)
cn = ["ctx_lower_stnum_next3", "ctx_st_back_count_w20", "ctx_st_threads_w20"]
Xc = np.column_stack([X] + [ctx[n] for n in cn]); alln = names + cn

pk = joblib.load(f"{ART}/ids_lgbm_ensemble_corrected.joblib")
qlo, qhi = pk["q_lo_ms"], pk["q_hi_ms"]; final = pk["feature_names"]
D = pd.DataFrame(Xc, columns=alln); ms = D["time_delta"].values * 1000
D["dt_in_baseline_flag"] = ((ms >= qlo) & (ms <= qhi)).astype(int)
D["dt_below_baseline"] = (ms < qlo).astype(int); D["dt_above_baseline"] = (ms > qhi).astype(int)
D["dt_in_pub_baseline_flag"] = 0; D["dt_above_pub_baseline"] = 0; D["dt_below_pub_baseline"] = 0

feats53 = list(final)
feats52 = [f for f in final if f != "ctx_lower_stnum_next3"]
print(f"rows={len(y):,}  features(53)={len(feats53)}  causal(52)={len(feats52)}")


## 1. Demonstrate the non-causality directly

For a row `i`, `ctx_lower_stnum_next3[i]` is 1 iff any of rows `i+1, i+2, i+3`
has a smaller `stNum`. We recompute it using **only past+current** rows and show
the two disagree — proving the published feature depends on future observations.

In [ ]:
st_arr = st
# causal recomputation (past+current only) cannot see i+1..i+3, so define the honest analogue:
prev_max20 = S.rolling(20, min_periods=1).max().shift(1).fillna(st_arr[0]).values
causal_analogue = (st_arr < prev_max20).astype(float)      # "lower than recent max" — trailing only
published = ctx["ctx_lower_stnum_next3"]

# pick a row where the published feature fires because of a FUTURE lower stNum
fire = np.flatnonzero(published == 1)
i = int(fire[len(fire)//2])
print(f"row i={i}:  stNum[i]={st_arr[i]:.0f}")
print(f"  next 3 stNum (i+1..i+3): {st_arr[i+1:i+4].astype(int).tolist()}  <- feature looks HERE (future)")
print(f"  ctx_lower_stnum_next3[i] = {published[i]:.0f}  (uses future rows)")
print(f"  a purely causal value at i cannot depend on i+1..i+3.")
print(f"\nAgreement between published (future-looking) and causal analogue: "
      f"{(published==causal_analogue).mean()*100:.1f}% of rows "
      f"({int((published!=causal_analogue).sum()):,} rows differ)")


## 2. Train/eval helpers (same target-recall policy for every configuration)

In [ ]:
idx = np.arange(len(y))
def split(cols):
    M = D[cols].values.astype(np.float32)
    Xa, Xte, ia, ite, ya, yte = train_test_split(M, idx, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
    Xtr, Xva, _, _, ytr, yva = train_test_split(Xa, ia, ya, test_size=VAL_SIZE, random_state=SEED, stratify=ya)
    return Xtr, Xva, Xte, ytr, yva, yte

def select_threshold(yv, pv):
    grid = np.unique(np.concatenate([np.linspace(0, 1, 1001), pv, [0.5]])); best = None
    for t in grid:
        pred = pv >= t; tp = int((pred & (yv == 1)).sum()); fn = int((~pred & (yv == 1)).sum())
        rec = tp / (tp + fn) if (tp + fn) else 0
        if rec >= VAL_FLOOR:
            fp = int((pred & (yv == 0)).sum()); k = (fp, -t)
            if best is None or k < best[0]: best = (k, t)
    return best[1] if best else 0.5

def cm(yt, sc, thr):
    p = (sc >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(yt, p).ravel(); return int(fp), int(fn), int(fp + fn)

def lgbm(Xtr, Xva, Xte, ytr, yva):
    spw = float((ytr == 0).sum() / (ytr == 1).sum())
    tr = lgb.Dataset(Xtr, ytr); va = lgb.Dataset(Xva, yva, reference=tr)
    m = lgb.train(dict(objective="binary", metric="average_precision", num_leaves=63, learning_rate=0.05,
                       scale_pos_weight=spw, verbose=-1, seed=11, num_threads=-1),
                  tr, num_boost_round=800, valid_sets=[va], callbacks=[lgb.early_stopping(50, verbose=False)])
    return m.predict(Xva, num_iteration=m.best_iteration), m.predict(Xte, num_iteration=m.best_iteration)

def xgbo(Xtr, Xva, Xte, ytr, yva):
    spw = float((ytr == 0).sum() / (ytr == 1).sum())
    m = xgb.XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
                          scale_pos_weight=spw, eval_metric="logloss", n_jobs=-1, random_state=SEED, tree_method="hist")
    m.fit(Xtr, ytr); return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]

def rfo(Xtr, Xva, Xte, ytr, yva):
    m = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, class_weight="balanced_subsample",
                               n_jobs=-1, random_state=SEED)
    m.fit(Xtr, ytr); return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]

MODELS = {"LightGBM": lgbm, "XGBoost": xgbo, "RandomForest": rfo}


## 3. Compare: 53-feature (non-causal) vs 52 causal vs 52 + causal replacement

In [ ]:
# causal replacement feature (trailing-only)
D["ctx_st_below_recent_max_w20"] = causal_analogue
feats_repl = feats52 + ["ctx_st_below_recent_max_w20"]
configs = {"53-feat (non-causal)": feats53, "52-feat (causal)": feats52, "52 + causal replacement": feats_repl}

rows = []
for mname, fn_ in MODELS.items():
    rec = {"model": mname}
    for cname, cols in configs.items():
        Xtr, Xva, Xte, ytr, yva, yte = split(cols)
        pv, pt = fn_(Xtr, Xva, Xte, ytr, yva)
        fp, fn2, err = cm(yte, pt, select_threshold(yva, pv))
        rec[cname] = f"{fp} FP / {fn2} FN = {err}"
    rows.append(rec)
res = pd.DataFrame(rows).set_index("model")
print(res.to_string()); res


## 4. Visualise the error impact

In [ ]:
import matplotlib.pyplot as plt
def errval(s): return int(s.split("=")[-1])
labels = list(MODELS); confs = list(configs)
colors = ["#D55E00", "#009E73", "#E69F00"]
xpos = np.arange(len(labels)); w = 0.26
fig, ax = plt.subplots(figsize=(9, 5))
for j, cname in enumerate(confs):
    vals = [errval(res.loc[m, cname]) for m in labels]
    ax.bar(xpos + (j-1)*w, vals, w, label=cname, color=colors[j])
    for x, v in zip(xpos + (j-1)*w, vals):
        ax.text(x, v + 0.4, str(v), ha="center", va="bottom", fontsize=8)
ax.set_xticks(xpos); ax.set_xticklabels(labels); ax.set_ylabel("Test errors (FP + FN)")
ax.set_title("Effect of removing the non-causal feature (target-recall policy, corrected split)")
ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("causal_feature_audit_errors.pdf", bbox_inches="tight"); plt.show()


## Conclusion

`ctx_lower_stnum_next3` depends on future rows and is therefore **not causal**.
Removing it roughly **doubles** the LightGBM error count and **triples** XGBoost's;
the trailing-only replacement recovers only part of the loss, because the first
injected packet of a block is *inherently* undetectable at its arrival instant —
it is confirmed only once the legitimate stream resumes.

Honest resolutions: **(1)** drop the feature and report the 52-feature causal
numbers as the online detector; **(2)** reframe it as an audit/explanation feature
(future-looking, like the TreeSHAP window), not a detection feature; or **(3)**
keep it but describe it as a bounded 3-packet look-ahead buffer — causal *with a
few-ms detection latency* — and report both the buffered and zero-latency results.
The current "zero-latency causal 53-feature detector" wording is not supported.